# Практична робота № 4


## Крок 1. Підготовка середовища та завантаження даних

In [1]:
import pandas as pd
import numpy as np

In [2]:
from google.colab import drive   # монтування Google Диска
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
PATH = '/content/drive/MyDrive/sales.csv'

# Якщо файлу sales.csv немає на Диску - створюємо навчальний набір про продажі кави
import os
if not os.path.exists(PATH):
    rng = np.random.default_rng(42)
    n = 600
    drinks = {"Latte": 38.7, "Americano": 28.9, "Cappuccino": 32.0, "Espresso": 25.0,
              "Hot Chocolate": 35.8, "Cocoa": 35.8, "Americano with Milk": 30.7, "Cortado": 27.0}
    names = rng.choice(list(drinks), n)
    cash = rng.choice(["card", "cash"], n, p=[0.9, 0.1])
    dt = pd.to_datetime("2024-03-01 08:00") + pd.to_timedelta(np.sort(rng.integers(0, 60*24*60, n)), unit="m")
    pd.DataFrame({
        "date": dt.date,
        "datetime": dt,
        "cash_type": cash,
        "card": [f"ANON-0000-0000-{i:04d}" if c == "card" else np.nan for i, c in enumerate(cash, 1)],
        "money": [round(drinks[x] * rng.choice([1.0, 1.0, 1.0, 1.1]), 2) for x in names],
        "coffee_name": names,
    }).to_csv(PATH, index=False)
    print("Створено навчальний файл:", PATH)

df = pd.read_csv(PATH)
df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')   # текст -> datetime64
df.head()

,date,datetime,cash_type,card,money,coffee_name
0,2024-03-01,2024-03-01 08:00:00,card,ANON-0000-0000-0000,24.0,Espresso
1,2024-03-01,2024-03-01 08:47:00,cash,ANON-0000-0000-0001,35.0,Cocoa
2,2024-03-01,2024-03-01 09:34:00,cash,ANON-0000-0000-0002,32.0,Cappuccino
3,2024-03-01,2024-03-01 10:21:00,card,ANON-0000-0000-0003,35.0,Cocoa
4,2024-03-01,2024-03-01 11:08:00,card,ANON-0000-0000-0004,35.0,Cocoa


## Крок 2. Групування за однією колонкою та агрегація

In [4]:
revenue_by_coffee = df.groupby('coffee_name')['money'].sum()   # дохід за напоями
print(revenue_by_coffee)

coffee_name
Americano        1213.8
Cappuccino       1664.0
Cocoa            1365.0
Espresso         1536.0
Hot Chocolate    1518.0
Latte            2205.9
Name: money, dtype: float64


In [5]:
avg_money = df.groupby('coffee_name')['money'].mean()          # середня сума транзакції
print(avg_money)

coffee_name
Americano        28.9
Cappuccino       32.0
Cocoa            35.0
Espresso         24.0
Hot Chocolate    33.0
Latte            38.7
Name: money, dtype: float64


## Крок 3. Групування за кількома колонками

In [6]:
trans_count = df.groupby(['date', 'coffee_name']).size()   # мультиіндексний Series
print(trans_count)

date        coffee_name  
2024-03-01  Americano        3
            Cappuccino       5
            Cocoa            6
            Espresso         4
            Hot Chocolate    3
                            ..
2024-03-10  Hot Chocolate    4
            Latte            8
2024-03-11  Cappuccino       1
            Espresso         1
            Hot Chocolate    1
Length: 62, dtype: int64


In [7]:
trans_count_df = df.groupby(['date', 'coffee_name']).size().reset_index(name='count')   # у звичайну таблицю
print(trans_count_df.head(10))

         date    coffee_name  count
0  2024-03-01      Americano      3
1  2024-03-01     Cappuccino      5
2  2024-03-01          Cocoa      6
3  2024-03-01       Espresso      4
4  2024-03-01  Hot Chocolate      3
5  2024-03-02      Americano      4
6  2024-03-02     Cappuccino      5
7  2024-03-02          Cocoa      3
8  2024-03-02       Espresso      8
9  2024-03-02  Hot Chocolate      5


## Крок 4. Кілька агрегатних функцій через `agg()`

In [8]:
summary = df.groupby('coffee_name')['money'].agg(['sum', 'mean', 'min', 'max'])
print(summary)

                  sum  mean   min   max
coffee_name                            
Americano      1213.8  28.9  28.9  28.9
Cappuccino     1664.0  32.0  32.0  32.0
Cocoa          1365.0  35.0  35.0  35.0
Espresso       1536.0  24.0  24.0  24.0
Hot Chocolate  1518.0  33.0  33.0  33.0
Latte          2205.9  38.7  38.7  38.7


## Крок 5. Зведена таблиця (`pivot_table`)

In [9]:
df['day_of_week'] = df['datetime'].dt.dayofweek   # 0 - понеділок, 6 - неділя

pivot = pd.pivot_table(df, values='money', index='coffee_name', columns='day_of_week', aggfunc='sum', fill_value=0)
print(pivot)

day_of_week        0      1      2      3      4      5      6
coffee_name                                                   
Americano       57.8  173.4   57.8   57.8  260.1  202.3  404.6
Cappuccino     192.0  192.0  128.0  192.0  320.0  416.0  224.0
Cocoa          210.0  140.0  280.0   70.0  315.0  175.0  175.0
Espresso       168.0   96.0  216.0   96.0  216.0  384.0  360.0
Hot Chocolate  132.0  132.0  198.0  198.0  231.0  297.0  330.0
Latte          348.3  232.2   77.4  425.7  270.9  464.4  387.0


## Крок 6. Перехресна таблиця (`crosstab`)

In [10]:
crosstab = pd.crosstab(df['coffee_name'], df['cash_type'])
print(crosstab)

cash_type      card  cash
coffee_name              
Americano        27    15
Cappuccino       41    11
Cocoa            28    11
Espresso         55     9
Hot Chocolate    38     8
Latte            50     7


## Крок 7. Об'єднання даних за допомогою `merge`

In [11]:
categories = pd.DataFrame({
    'coffee_name': ['Latte', 'Americano', 'Hot Chocolate', 'Cappuccino'],
    'category': ['молочний', 'чорний', 'молочний', 'молочний']
})

df_merged = df.merge(categories, on='coffee_name', how='left')   # лівий join: усі рядки df зберігаються
print(df_merged.head())

         date            datetime cash_type                 card  money  \
0  2024-03-01 2024-03-01 08:00:00      card  ANON-0000-0000-0000   24.0   
1  2024-03-01 2024-03-01 08:47:00      cash  ANON-0000-0000-0001   35.0   
2  2024-03-01 2024-03-01 09:34:00      cash  ANON-0000-0000-0002   32.0   
3  2024-03-01 2024-03-01 10:21:00      card  ANON-0000-0000-0003   35.0   
4  2024-03-01 2024-03-01 11:08:00      card  ANON-0000-0000-0004   35.0   

  coffee_name  day_of_week  category  
0    Espresso            4       NaN  
1       Cocoa            4       NaN  
2  Cappuccino            4  молочний  
3       Cocoa            4       NaN  
4       Cocoa            4       NaN  


In [12]:
print("Рядків без категорії (NaN):", df_merged['category'].isnull().sum())

Рядків без категорії (NaN): 103


## Крок 8. Об'єднання за допомогою `concat`

In [13]:
df_part1 = df[df['date'] <= '2024-03-15']
df_part2 = df[df['date'] > '2024-03-15']

df_concat = pd.concat([df_part1, df_part2])
print(f"Розміри: {len(df_part1)} + {len(df_part2)} = {len(df_concat)}")

Розміри: 300 + 0 = 300


## Крок 9. Об'єднання за допомогою `join`

In [14]:
left = pd.DataFrame({'A': [1, 2, 3]}, index=['a', 'b', 'c'])
right = pd.DataFrame({'B': [10, 20, 30]}, index=['a', 'b', 'c'])

joined = left.join(right)   # за замовчуванням - ліве об'єднання за індексами
print(joined)

   A   B
a  1  10
b  2  20
c  3  30


## Крок 10. Аналітичні показники за групами: частка доходу

In [15]:
total_revenue = df['money'].sum()
revenue_share = (df.groupby('coffee_name')['money'].sum() / total_revenue * 100).round(2)
print(revenue_share)

coffee_name
Americano        12.77
Cappuccino       17.51
Cocoa            14.36
Espresso         16.16
Hot Chocolate    15.97
Latte            23.21
Name: money, dtype: float64
